# load the cleaned data and analyse

Analyze customers by age group, gender, and premium status.
Which group has more customers?

In [1]:
import pandas as pd

In [2]:
c_df=pd.read_csv('cleaned_datasets/customer.csv')

In [3]:
p_df=pd.read_json('cleaned_datasets/product.json')

In [4]:
o_df=pd.read_excel('cleaned_datasets/orders.xlsx')

1. Analyze customers by age group,gender and premium status. Which group has more customers?

In [5]:
c_df['age_category'].value_counts()

age_category
Adult       432
Teenage     121
Children     16
Name: count, dtype: int64

In [6]:
c_df['gender'].value_counts()

gender
Female    351
Male      218
Name: count, dtype: int64

In [7]:
c_df['premium_status'].value_counts()

premium_status
1    286
0    283
Name: count, dtype: int64

In [8]:
c_df.groupby(by='premium_status').agg({'premium_status':'count'})

,premium_status
premium_status,
0,283
1,286


Observation
1. Adults customers are more
         2.Female customers are more
3.Premium status are more

2.Show number of orders over time (month-wise or year-wise).
Is the number of orders increasing or decreasing?

In [9]:
o_df['order_date'].dt.year.value_counts().sort_index()   #dt is an attribute using this we can access datetimesort_index()

order_date
2023    120
2024    177
2025    176
2026     19
Name: count, dtype: int64

In [10]:
o_df['order_date'].dt.month.value_counts().sort_index()

order_date
1     48
2     34
3     33
4     21
5     53
6     42
7     42
8     50
9     52
10    44
11    33
12    40
Name: count, dtype: int64

Observation

No of orders slightly increase over a period of time[year wise] consider this

No of orders fluctuating over a period of time[month wise]

3. Compare no_of orders for:

(i)Premium vs non-premium customers

(ii)Male vs female customers

the question on orders and customer we need to merge

In [11]:
temp=pd.merge(left=c_df[['customer_id','gender','premium_status']],right=o_df[['customer_id','order_date']])

In [12]:
# (i)Premium vs non-premium customers

temp.groupby(by='premium_status').agg({ 'order_date': 'count' })

,order_date
premium_status,
0,238
1,254


In [13]:
temp.groupby(by=['gender']).agg({'order_date':'count'})

,order_date
gender,
Female,285
Male,207


4.find revenue for each sold product category.
Which category gives the highest revenue?

In [14]:
temp=pd.merge(left=p_df[['product_id','category']],right=o_df[['product_id','amount_to_be_paid']])

In [15]:
temp.groupby(by='category').agg({'amount_to_be_paid' :'sum'}).sort_values(by='amount_to_be_paid', ascending=False)

,amount_to_be_paid
category,
Electronics,409783.15
Accessories,134470.00
Hygiene,130125.50
Footwear,129704.00
Furniture,120810.00
Grocery,45614.40


5. Find average rating for each product category.
Which categories have low ratings?

In [16]:
p_df.groupby(by='category').agg({'ratings': 'mean'})

,ratings
category,
Accessories,2.900000
Electronics,3.088235
Footwear,3.545455
Furniture,3.333333
Grocery,3.750000
Hygiene,3.111111


In [17]:
p_df.groupby(by='category').agg({'ratings': 'mean'}).idxmin()

ratings    Accessories
dtype: str

6.Identify repeat customers and one-time customers.
How many customers belong to each group?

In [18]:
temp=o_df['customer_id'].value_counts()

In [19]:
temp==1

customer_id
C346    False
C175    False
C601    False
C477    False
C266    False
        ...  
C199     True
C874     True
C436     True
C645     True
C993     True
Name: count, Length: 311, dtype: bool

In [20]:
temp[temp==1]
# only true data is selected

customer_id
C415    1
C534    1
C291    1
C856    1
C239    1
       ..
C199    1
C874    1
C436    1
C645    1
C993    1
Name: count, Length: 191, dtype: int64

In [21]:
temp[temp==1].shape
# ==1 exactly once
# >1 repeated

(191,)

In [22]:
temp[temp>1].shape

(120,)

**observation**

Out of 311 no_of customers , 120 customers are repeated customers and 191 customers are one-tine customers

Calculate percentage of total revenue from repeat customers.
Are repeat customers important for the business?

In [23]:
o_df.shape
# 492 rows and 9 columns

(492, 9)

In [24]:
temp=o_df['customer_id'].value_counts()
cid=temp[temp>1].index

print(cid)
# when value count is used, the column name given inside the square bracket (group by) act as index 

Index(['C346', 'C175', 'C601', 'C477', 'C266', 'C494', 'C254', 'C978', 'C969',
       'C892',
       ...
       'C377', 'C576', 'C690', 'C490', 'C903', 'C126', 'C770', 'C778', 'C788',
       'C380'],
      dtype='str', name='customer_id', length=120)


In [25]:
o_df['customer_id'].isin(cid)
# if we need only true value use "loc " attribute
# to check multiple values we can use "in", in pandas in is not supported so we use :inuse
# isin also give boolean values from this d.loc take only true values

0      False
1      False
2      False
3       True
4      False
       ...  
487    False
488     True
489     True
490     True
491     True
Name: customer_id, Length: 492, dtype: bool

In [26]:
o_df.loc[o_df['customer_id'].isin(cid)]

,order_id,customer_id,product_id,order_date,net_quantity,payment_mode,amount_saved,actual_total,amount_to_be_paid
3,O5003,C650,P1090,2023-05-21,5,cash,5000.00,6750,1750.00
7,O5007,C933,P1090,2024-12-25,4,net banking,4000.00,5400,1400.00
9,O5009,C540,P1097,2025-03-15,5,cash,0.00,2625,2625.00
10,O5010,C238,P1095,2023-10-01,1,debit card,0.00,100,100.00
12,O5012,C266,P1029,2025-02-04,2,credit card,335.40,2236,1900.60
...,...,...,...,...,...,...,...,...,...
486,O5494,C638,P1044,2023-05-09,3,debit card,405.00,4050,3645.00
488,O5496,C970,P1087,2023-07-13,3,credit card,236.25,1575,1338.75
489,O5497,C657,P1036,2023-07-23,5,debit card,5000.00,5590,590.00
490,O5498,C257,P1022,2024-10-30,2,credit card,30.00,200,170.00


In [27]:
o_df.loc[o_df['customer_id'].isin(cid),'amount_to_be_paid']

3      1750.00
7      1400.00
9      2625.00
10      100.00
12     1900.60
        ...   
486    3645.00
488    1338.75
489     590.00
490     170.00
491   -1900.00
Name: amount_to_be_paid, Length: 301, dtype: float64

In [28]:
# for repeated
repeated_cus_total_revenue=o_df.loc[o_df['customer_id'].isin(cid),'amount_to_be_paid'].sum()

In [29]:
# all customers repeated and repeated
total_revenue=o_df['amount_to_be_paid']
total_revenue

0      4080.00
1      5737.50
2       400.00
3      1750.00
4      6750.00
        ...   
487    5400.00
488    1338.75
489     590.00
490     170.00
491   -1900.00
Name: amount_to_be_paid, Length: 492, dtype: float64

In [30]:
repeated_cus_total_revenue/total_revenue*100

0       13796.827206
1        9811.077124
2      140727.637500
3       32166.317143
4        8339.415556
           ...      
487     10424.269444
488     42047.473389
489     95408.567797
490    331123.852941
491    -29626.871053
Name: amount_to_be_paid, Length: 492, dtype: float64

In [31]:
# to reduce the round values
(repeated_cus_total_revenue/total_revenue*100).round()

0       13797.0
1        9811.0
2      140728.0
3       32166.0
4        8339.0
         ...   
487     10424.0
488     42047.0
489     95409.0
490    331124.0
491    -29627.0
Name: amount_to_be_paid, Length: 492, dtype: float64

**observation**

repeated customers have contributed 58% of total revenue

Analyze premium customer behavior:

(i)How often they place orders

(ii)How much they spend on average

(iii)Which payment mode they use most

In [32]:
# (i)How often they place orders
# pd.merge(c_df,o_df,on='customer_id',how='inner')
df1=pd.merge(c_df[['customer_id','premium_status']],o_df[['customer_id','payment_mode','amount_to_be_paid']],on='customer_id',how='inner')
df1
# 491 rows

,customer_id,premium_status,payment_mode,amount_to_be_paid
0,C754,0,net banking,2850.9
1,C132,0,credit card,3060.0
2,C132,0,credit card,3018.6
3,C132,0,debit card,4590.0
4,C649,1,credit card,-1900.0
...,...,...,...,...
487,C259,0,debit card,2400.0
488,C805,0,net banking,4472.0
489,C425,1,cash,300.0
490,C410,0,cash,525.0


In [33]:
df1.loc[df1['premium_status']==1]
# premium_status has only 1 in every row no need to it.
premium_df=df1.loc[df1['premium_status']==1,['customer_id','payment_mode','amount_to_be_paid']]

In [34]:
premium_df['customer_id'].value_counts()

customer_id
C601    6
C266    4
C494    4
C254    4
C240    4
       ..
C752    1
C121    1
C734    1
C533    1
C425    1
Name: count, Length: 161, dtype: int64

In [35]:
premium_df['amount_to_be_paid'].mean()

np.float64(1920.328937007874)

**observation**
the premium customer C601 has placed the orders for 6 times , which is the highest orders among all premium customers.

the avg amount spent by the premium customers is 1920.14

the most preferd payment mode is debit card

In [36]:
# (ii)How much they spend on average
premium_df['payment_mode'].mode()

0    debit card
Name: payment_mode, dtype: str

In [37]:
premium_df['payment_mode'].unique()

<StringArray>
['credit card', 'cash', 'upi', 'debit card', 'net banking']
Length: 5, dtype: str

8.Check if high-selling products are frequently out of stock.
Is stock availability affecting sales?

In [38]:
p_df.columns

Index(['product_id', 'price', 'discount', 'expiry_date', 'stock', 'ratings',
       'discounted amount', 'amazon', 'flipkart', 'product', 'category'],
      dtype='str')

In [39]:
df2=pd.merge(p_df[['product_id','stock','ratings']],o_df[['product_id','amount_to_be_paid']], how='inner', on='product_id')
df2

,product_id,stock,ratings,amount_to_be_paid
0,P1003,available,2,1400.0
1,P1003,available,2,1050.0
2,P1003,available,2,350.0
3,P1003,available,2,1750.0
4,P1003,available,2,1400.0
...,...,...,...,...
487,P1098,available,2,4080.0
488,P1098,available,2,5100.0
489,P1098,available,2,1020.0
490,P1098,available,2,1020.0


In [40]:
df2.loc[df2['stock']!='available']

,product_id,stock,ratings,amount_to_be_paid


9.Compare product ratings vs sales.
Do high-rated products sell more than low-rated ones?

In [41]:
df2=pd.merge(p_df[['product_id','ratings','discounted amount']],o_df[['product_id','amount_to_be_paid']], how='inner', on='product_id')
df2[['ratings','amount_to_be_paid','discounted amount']].corr()
# if correlation is 0 means no relation

,ratings,amount_to_be_paid,discounted amount
ratings,1.000000,0.043991,0.084105
amount_to_be_paid,0.043991,1.000000,0.797005
discounted amount,0.084105,0.797005,1.000000


**observation**

ratings and sales amount are not related

10.Identify products with low sales and low ratings.
Which products should be discontinued from a business point of view?

In [42]:
# least sales and least rating

11.Based on customer behavior, identify top 3 customer segments the business should focus on.


Based on sales, ratings, and stock:

(i)Which products should be promoted?

(ii)Which products should be discounted?

(iii)Which products should be discontinued?

In [43]:

# 1. focus on sales and rating because stock is available in evry row

12.Find total spending of premium and non-premium customers

In [44]:
c_df

,customer_id,age,gender,occupation,married,registered_date,premium_status,age_category
0,C754,21,Female,Student,1,2023-04-22,0,Teenage
1,C854,20,Female,Teacher,1,2019-03-10,0,Teenage
2,C132,17,Male,Doctor,1,2019-07-08,0,Children
3,C833,43,Female,Farmer,1,2020-10-09,1,Adult
4,C928,52,Female,Unemployed,1,2017-12-24,0,Adult
...,...,...,...,...,...,...,...,...
564,C686,23,Male,Teacher,0,2020-10-09,0,Teenage
565,C805,46,Female,Doctor,0,2024-03-07,0,Adult
566,C425,19,Female,Business,1,2023-09-29,1,Teenage
567,C410,35,Female,Business,0,2024-02-08,0,Adult


In [45]:
o_df

,order_id,customer_id,product_id,order_date,net_quantity,payment_mode,amount_saved,actual_total,amount_to_be_paid
0,O5000,C415,P1076,2025-01-06,4,credit card,720.00,4800,4080.00
1,O5001,C534,P1012,2025-09-03,5,cash,1012.50,6750,5737.50
2,O5002,C291,P1069,2025-04-09,4,credit card,0.00,400,400.00
3,O5003,C650,P1090,2023-05-21,5,cash,5000.00,6750,1750.00
4,O5004,C856,P1019,2023-07-05,5,debit card,0.00,6750,6750.00
...,...,...,...,...,...,...,...,...,...
487,O5495,C993,P1024,2023-08-06,5,credit card,600.00,6000,5400.00
488,O5496,C970,P1087,2023-07-13,3,credit card,236.25,1575,1338.75
489,O5497,C657,P1036,2023-07-23,5,debit card,5000.00,5590,590.00
490,O5498,C257,P1022,2024-10-30,2,credit card,30.00,200,170.00


In [46]:
merged_df = o_df.merge(c_df, on='customer_id')

# Total spending by premium vs non-premium
result = merged_df.groupby('premium_status')['amount_to_be_paid'].sum()

print(result)

premium_status
0    482743.50
1    487763.55
Name: amount_to_be_paid, dtype: float64


13.Find total sales for available vs not available products.

In [47]:
merged_df = o_df.merge(p_df, on='product_id')

# Group directly using stock column
result = merged_df.groupby('stock')['amount_to_be_paid'].sum().reset_index()

print(result)

       stock  amount_to_be_paid
0  available          970507.05


14.Find which customer type is buying products with low ratings and suggest one business improvement.

In [48]:
print(p_df.columns.tolist())

['product_id', 'price', 'discount', 'expiry_date', 'stock', 'ratings', 'discounted amount', 'amazon', 'flipkart', 'product', 'category']


In [49]:
o_df = pd.merge(
    o_df,
    p_df[['product_id', 'ratings']],
    on='product_id',
    how='left'
)

In [50]:
# promoted
promo = o_df[(o_df['ratings'] > 4) & 
             (o_df['amount_to_be_paid'] < o_df['amount_to_be_paid'].mean())]

promo[['product_id']].drop_duplicates()

,product_id
113,P1056
302,P1045


In [51]:
#discount
discount = o_df[(o_df['amount_to_be_paid'] > 0) & 
                (o_df['amount_to_be_paid'] < o_df['amount_to_be_paid'].mean())]

discount[['product_id']].drop_duplicates()

,product_id
2,P1069
3,P1090
6,P1017
8,P1009
10,P1095
12,P1029
13,P1042
14,P1036
15,P1016
16,P1022


In [52]:
#discontinued
discontinue = o_df[(o_df['ratings'] < 3) & 
                   (o_df['amount_to_be_paid'] < o_df['amount_to_be_paid'].mean())]

discontinue[['product_id']].drop_duplicates()

,product_id
8,P1009
13,P1042
24,P1003
68,P1005
70,P1011
109,P1020
143,P1082
179,P1004
253,P1098
309,P1019


Find total spending of premium and non-premium customers

In [53]:
temp = pd.merge(c_df[['customer_id','premium_status']],
                o_df[['customer_id','amount_to_be_paid']],
                on='customer_id')

temp.groupby('premium_status')['amount_to_be_paid'].sum()

premium_status
0    482743.50
1    487763.55
Name: amount_to_be_paid, dtype: float64

Find total sales for available vs not available products.


In [54]:
p_df['stock'].unique()

<StringArray>
['not available', 'available']
Length: 2, dtype: str

In [55]:
temp = pd.merge(p_df[['product_id','stock']],
                o_df[['product_id','amount_to_be_paid']],
                on='product_id')

temp['availability'] = temp['stock'].apply(lambda x: 'Available' if x == 'available' else 'Not Available')

temp.groupby('availability')['amount_to_be_paid'].sum()

availability
Available    970507.05
Name: amount_to_be_paid, dtype: float64

Find which customer type is buying products with low ratings and suggest one business improvement.


In [56]:
temp = pd.merge(o_df[['customer_id','product_id','ratings']],
                c_df[['customer_id','premium_status']],
                on='customer_id')

low_rating = temp[temp['ratings'] < 3]

low_rating['premium_status'].value_counts()

premium_status
1    60
0    53
Name: count, dtype: int64